# AI Enabled Smart Helmet for Drowsiness and Accident Detection
**Project ID:** 24AB039  
**Team:** Manas J Prince, Fiya Sulfikkar, Sidharth S, Alen Jose M S  
**Supervisor:** Dhanusha P B

## Software demonstration (Google Colab)
This notebook recreates the synthetic dataset-generation steps from the original project notebook and runs an **existing Edge Impulse-trained** accident/normal classifier. Drowsiness, buzzer and emergency alert are **software simulations**. No physical sensors, SMS, GPS or emergency services are connected.

**Before running:** obtain the exported **Linux x86_64 `.eim`** model and the Edge Impulse test sample **`accident_test.s16.json`** from your own project. Run cells in order in a Google Colab Linux runtime. The notebook does **not** retrain the model; the generated CSVs alone do not reproduce the existing model or its exact Edge Impulse train/test window split.

**Reported Edge Impulse evaluation:** 34/35 = **97.14%** on synthetic test samples; not real-world crash accuracy.

## 1. Generate synthetic accelerometer recordings
These cells preserve the original dataset-generation and 80/20 CSV split logic. Readings use a gravity baseline of about **9.8 m/s²**. The generated files are illustrative synthetic data.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Sampling frequency: 10 Hz
# Total duration: 180 seconds per class
time = np.arange(0, 180000, 100)
n = len(time)

# NORMAL MOVEMENT
normal_x = np.random.normal(0, 0.7, n)
normal_y = np.random.normal(0, 0.7, n)
normal_z = np.random.normal(9.8, 0.7, n)

# Add gentle movement variations
normal_x += 0.5 * np.sin(np.linspace(0, 30, n))
normal_y += 0.4 * np.sin(np.linspace(0, 20, n))

normal_data = pd.DataFrame({
    "timestamp": time,
    "accX": normal_x,
    "accY": normal_y,
    "accZ": normal_z
})

normal_data.to_csv("normal_medium.csv", index=False)

# ACCIDENT-LIKE MOVEMENT
accident_x = np.random.normal(0, 1.2, n)
accident_y = np.random.normal(0, 1.2, n)
accident_z = np.random.normal(9.8, 1.2, n)

# Add sudden acceleration changes
for i in range(10, n - 10, 20):
    spike = np.random.uniform(12, 25)
    accident_x[i:i+3] += spike
    accident_y[i:i+3] -= spike * 0.6
    accident_z[i:i+3] += spike * 0.4

accident_data = pd.DataFrame({
    "timestamp": time,
    "accX": accident_x,
    "accY": accident_y,
    "accZ": accident_z
})

accident_data.to_csv("accident_medium.csv", index=False)

print("Datasets created successfully!")
print("Normal readings:", len(normal_data))
print("Accident-like readings:", len(accident_data))
print("Total readings:", len(normal_data) + len(accident_data))

In [ ]:
import pandas as pd

# Load the generated datasets
normal = pd.read_csv("normal_medium.csv")
accident = pd.read_csv("accident_medium.csv")

# Split at 144 seconds (80%)
split_index = 1440

datasets = {
    "normal_train.csv": normal.iloc[:split_index],
    "normal_test.csv": normal.iloc[split_index:],
    "accident_train.csv": accident.iloc[:split_index],
    "accident_test.csv": accident.iloc[split_index:]
}

for filename, df in datasets.items():
    df = df.copy()

    # Reset timestamps to start at zero
    df["timestamp"] = (
        df["timestamp"] - df["timestamp"].iloc[0]
    )

    df.to_csv(filename, index=False)

    print(filename, "-", len(df), "readings")

## 2. Install inference dependencies
Run this once per new Colab runtime. Installation may take a minute.

In [ ]:
!apt-get install -y -qq portaudio19-dev
%pip install -q edge-impulse-linux pyaudio

## 3. Load your exported Edge Impulse model
Upload the Linux `.eim` executable if it is not already present in `/content`. Keep this file private if your project requires it; no API key is needed for local inference.

In [ ]:
from google.colab import files
from edge_impulse_linux.runner import ImpulseRunner
import glob, os

model_files = glob.glob('/content/*.eim')
if not model_files:
    print('Upload your Edge Impulse Linux x86_64 .eim file:')
    files.upload()
    model_files = glob.glob('/content/*.eim')
if not model_files:
    raise FileNotFoundError('No .eim model found. Upload the Linux model and rerun this cell.')
model_path = os.path.abspath(model_files[0])
os.chmod(model_path, 0o755)
runner = ImpulseRunner(model_path)
model_info = runner.init()
print('Model connected:', os.path.basename(model_path))
print('Classes:', model_info['model_parameters']['labels'])
print('Input values:', model_info['model_parameters']['input_features_count'])

## 4. Load a known synthetic accident test sample
This is an exported sample from the **Edge Impulse test set**. The original JSON has 21 readings; inference here uses the **first 20 × 3 = 60 values**, matching the original successful Colab demonstration.

In [ ]:
import json, numpy as np, glob
json_files = glob.glob('/content/accident_test.s16*.json')
if not json_files:
    print('Upload accident_test.s16.json:')
    files.upload()
    json_files = glob.glob('/content/accident_test.s16*.json')
if not json_files:
    raise FileNotFoundError('accident_test.s16.json is required for this demonstration.')
with open(json_files[0], encoding='utf-8') as f:
    sample = json.load(f)
readings = np.asarray(sample['payload']['values'], dtype=float)
if readings.ndim != 2 or readings.shape[1] != 3 or len(readings) < 20:
    raise ValueError('Expected at least 20 three-axis accelerometer readings.')
accident_input = readings[:20].flatten().tolist()
print('Loaded accident sample:', len(readings), 'readings; using', len(accident_input), 'values')
print('Accident prediction:', runner.classify(accident_input)['result']['classification'])

## 5. Interactive safety demonstration
- **Normal riding:** synthetic three-axis input sent to the trained model.
- **Accident:** saved synthetic accident sample sent to the trained model.
- **Drowsiness:** preset 4-second eye closure compared with a 3-second threshold (**not ML**).
- **Emergency alert:** on-screen only; **I’M SAFE** cancels the displayed alert.

The selected scenario determines which test input is supplied, **not** the model prediction. Predictions depend on the model output. **Caution:** the original notebook used ~1g normal demonstration values, while its generated CSV recordings use ~9.8 m/s². The different input scales are preserved here for fidelity; consistent units and independent validation are required before claiming reliable performance.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

rng = np.random.default_rng(42)
normal_input = np.column_stack([
    rng.normal(0, 0.05, 20),
    rng.normal(0, 0.05, 20),
    rng.normal(1, 0.05, 20)
]).flatten().tolist()  # Original dashboard demonstration input (~1g)

scenario = widgets.Dropdown(options=['Normal Riding', 'Drowsiness', 'Accident'], description='Scenario:')
check_button = widgets.Button(description='CHECK HELMET STATUS', button_style='primary')
safe_button = widgets.Button(description="I'M SAFE - CANCEL ALERT", button_style='success', disabled=True)
output = widgets.Output()
state = {'emergency_active': False}

def check_status(_):
    selected = scenario.value
    state['emergency_active'] = False
    safe_button.disabled = True
    with output:
        clear_output(wait=True)
        print('========== AI ENABLED SMART HELMET ==========')
        print('Scenario:', selected)
        if selected == 'Drowsiness':
            eye_closed_seconds = 4  # Simulated; no IR sensor connected
            print('Eye closure (simulated):', eye_closed_seconds, 'seconds')
            print('Detection method: 3-second threshold (not AI)')
            if eye_closed_seconds >= 3:
                print('DROWSINESS WARNING | BUZZER: ON (simulated)')
            else:
                print('NORMAL | BUZZER: OFF')
        else:
            inputs = accident_input if selected == 'Accident' else normal_input
            scores = runner.classify(inputs)['result']['classification']
            print(f"Accident score: {scores['accident'] * 100:.2f}%")
            print(f"Normal score: {scores['normal'] * 100:.2f}%")
            if scores['accident'] > scores['normal']:
                print('ACCIDENT-LIKE PATTERN DETECTED | BUZZER: ON (simulated)')
                print('EMERGENCY ALERT: ACTIVE (ON-SCREEN SIMULATION ONLY)')
                state['emergency_active'] = True
                safe_button.disabled = False
            else:
                print('NORMAL RIDING | BUZZER: OFF')

def cancel_alert(_):
    if not state['emergency_active']:
        return
    state['emergency_active'] = False
    safe_button.disabled = True
    with output:
        clear_output(wait=True)
        print('RIDER CONFIRMED SAFE (BUTTON PRESSED)')
        print('ON-SCREEN EMERGENCY ALERT: CANCELLED')
        print('BUZZER: OFF (simulated)')

check_button.on_click(check_status)
safe_button.on_click(cancel_alert)
display(scenario, check_button, safe_button, output)

## 6. Results and limitations
- In the original Edge Impulse project, **34 of 35 synthetic test windows** were correctly classified (97.14%).
- One synthetic accident test sample was misclassified as normal.
- The previously tested `accident_test.s16.json` produced **99.61% accident score** with the exported model; scores can vary with model/version or input handling.
- The normal and accident demo inputs are simulated and preselected; drowsiness is a rule, not a trained model.
- The alarm, cancellation and emergency alert are on-screen simulations only.
- There is **no physical ESP32, accelerometer, eye sensor or buzzer integration**, and no real-world accident validation.
- Synthetic windows may be correlated; do not interpret synthetic-test accuracy as field performance.

### Future work
Collect safe, real-world riding data; check sensor units and sample windows; compare against a simple threshold baseline; test potholes/braking/helmet drops; then consider ESP32 deployment and responsibly designed contact notifications.